# SpaGE imputation on Visium HD — reference comparison

Compare SpaGE imputation across SIMPlex and BC Atlas references on a Visium HD ROI.

**Workflow**
1. Load spatial ROI + reference snRNA (all from Seurat RDS via R)
2. SpaGE imputation per reference cohort
3. Spatial maps (measured vs imputed) → `figs/spaGE/`
4. Spearman concordance (imputed vs measured)

**Samples:** `patient4_HD` or `patient5_HD`. Section-matched snRNA = `sample == sample_id` in `SIMPlex_breast_allSamples.rds`.  
Default case study: `patient5_HD` ↔ BC atlas `CID44971` (same cohort labels as [`extra_exploration.ipynb`](extra_exploration.ipynb)).


In [ ]:
import subprocess
import tempfile
import warnings
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.stats as st
import seaborn as sns
from scipy.io import mmread
from sklearn.neighbors import NearestNeighbors
from SpaGE.principal_vectors import PVComputation

warnings.filterwarnings("ignore")

REPO = Path("/srv/home/m.abreumachado/git_repos/SIMPlex_analysis")
SPACERANGER = REPO / "data/spatial/spaceranger"
FIG_OUT = REPO / "figs/spaGE"
FIG_OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "figure.dpi": 150,
})

In [ ]:
# --- Configuration ---
sample_id = "patient5_HD"  # or "patient4_HD"

SAMPLES = {
    "patient4_HD": {
        "spatial_h5": SPACERANGER / "breast_cancer/HD/patient4_HD/binned_outputs/square_008um/filtered_feature_bc_matrix.h5",
"tissue_positions": SPACERANGER / "breast_cancer/HD/patient4_HD/binned_outputs/square_008um/spatial/tissue_positions.parquet",
        "patient": "patient4",
        "atlas_patient": None,
    },
    "patient5_HD": {
        "spatial_h5": SPACERANGER / "breast_cancer/HD/patient5_HD/binned_outputs/square_008um/filtered_feature_bc_matrix.h5",
"tissue_positions": SPACERANGER / "breast_cancer/HD/patient5_HD/binned_outputs/square_008um/spatial/tissue_positions.parquet",
        "patient": "patient5",
        "atlas_patient": "CID44971",
    },
}

ROI_X_FRAC = (0.40, 0.60)
ROI_Y_FRAC = (0.40, 0.60)
N_PV = 30
MAX_ALIGN_GENES = 3000

COHORT = {
    "simp_section": "SIMPlex (section-matched)",
    "simp_other": "SIMPlex (other sections)",
    "simp_pooled": "SIMPlex (pooled)",
    "bc_patient": "BC Atlas (patient-matched)",
    "bc_all": "BC Atlas (all patients)",
}
COHORT_KEYS = list(COHORT.keys())
COHORT_LABELS = list(COHORT.values())
COHORT_COLORS = {
    "SIMPlex (section-matched)": "#6A5ACD",
    "SIMPlex (other sections)": "#9370DB",
    "SIMPlex (pooled)": "#B19CD9",
    "BC Atlas (patient-matched)": "#20B2AA",
    "BC Atlas (all patients)": "#E89550",
}

PANEL_GENES = [
    "EPCAM", "KRT8", "CD3D", "CD3E", "COL1A1", "MKI67", "CD68", "PECAM1",
    "MS4A1", "FAP", "MUC1", "IGKC", "IGHG1",
]

PATIENT = SAMPLES[sample_id]["patient"]
ATLAS_PATIENT = SAMPLES[sample_id]["atlas_patient"]
RDS_REF_KEYS = tuple(COHORT_KEYS)  # all references exported from R

In [ ]:
# --- Functions ---
R_LOAD_REFS = r"""
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
})
args <- commandArgs(trailingOnly = TRUE)
out_dir <- args[[1]]
patient_id <- args[[2]]
atlas_patient_id <- args[[3]]
repo_root <- args[[4]]
spatial_sample_id <- args[[5]]

SN_RDS <- file.path(repo_root, "data/single_nuclei/r_objects")
EXT_REFS <- file.path(repo_root, "data/other/external_references")

save_ref_counts <- function(mat, prefix) {
  mat <- mat[!grepl("^DEPRECATED_", rownames(mat)), , drop = FALSE]
  writeLines(rownames(mat), paste0(prefix, "_genes.txt"))
  writeLines(colnames(mat), paste0(prefix, "_barcodes.txt"))
  Matrix::writeMM(mat, paste0(prefix, "_matrix.mtx"))
  message("Wrote ", prefix, " (", nrow(mat), " genes x ", ncol(mat), " cells)")
}

get_counts <- function(obj, assay = "RNA") {
  DefaultAssay(obj) <- assay
  if (inherits(obj[[assay]], "Assay5")) {
    obj <- tryCatch(JoinLayers(obj, assay = assay), error = function(e) obj)
  }
  tryCatch(LayerData(obj, layer = "counts"), error = function(e) GetAssayData(obj, slot = "counts"))
}

message("Loading SIMPlex object ...")
int.all <- readRDS(file.path(SN_RDS, "breast_cancer/cross_patient/SIMPlex_breast_allSamples.rds"))
if (!"patient_ID" %in% colnames(int.all@meta.data)) {
  int.all$patient_ID <- sub("_.*$", "", as.character(int.all$sample))
}

message("Loading BC atlas ...")
bc_atlas <- readRDS(file.path(EXT_REFS, "BC_atlas/miniatlas.rds"))
bc_atlas <- UpdateSeuratObject(bc_atlas)
bc_atlas$celltype_major <- ifelse(
  bc_atlas$celltype_major %in% c("Normal Epithelial", "Cancer Epithelial"),
  "Epithelial", bc_atlas$celltype_major
)

refs <- list(
  simp_section = subset(int.all, subset = sample == spatial_sample_id),
  simp_other = subset(int.all, subset = patient_ID != patient_id),
  simp_pooled = int.all,
  bc_patient = subset(bc_atlas, subset = Patient == atlas_patient_id),
  bc_all = bc_atlas
)

for (nm in names(refs)) {
  save_ref_counts(get_counts(refs[[nm]]), file.path(out_dir, nm))
}
message("Done.")
"""

_RNA_REFS = {}


def filter_gene_columns(df):
    keep = [g for g in df.columns if not str(g).startswith("DEPRECATED_")]
    return df.loc[:, keep]


def dedupe_columns(df):
    if not df.columns.duplicated().any():
        return df
    return df.T.groupby(level=0).sum().T


def read_10x_h5(path, barcodes=None):
    with h5py.File(path, "r") as f:
        m = f["matrix"]
        all_barcodes = [x.decode() if isinstance(x, bytes) else x for x in m["barcodes"][:]]
        genes = [x.decode() if isinstance(x, bytes) else x for x in m["features"]["name"][:]]
        X = sp.csc_matrix(
            (m["data"][:], m["indices"][:], m["indptr"][:]),
            shape=tuple(m["shape"][:]),
        )
        if barcodes is not None:
            barcode_to_idx = {bc: i for i, bc in enumerate(all_barcodes)}
            cols = [barcode_to_idx[bc] for bc in barcodes if bc in barcode_to_idx]
            X = X[:, cols]
            all_barcodes = [all_barcodes[i] for i in cols]
    return filter_gene_columns(dedupe_columns(pd.DataFrame(X.toarray().T, index=all_barcodes, columns=genes)))


def read_mtx_ref(prefix):
    prefix = Path(prefix)
    genes = Path(f"{prefix}_genes.txt").read_text().splitlines()
    cells = Path(f"{prefix}_barcodes.txt").read_text().splitlines()
    mat = mmread(f"{prefix}_matrix.mtx").tocsc()
    return filter_gene_columns(dedupe_columns(pd.DataFrame(mat.toarray().T, index=cells, columns=genes)))


def load_rds_refs_from_r(patient=PATIENT, atlas_patient=ATLAS_PATIENT, spatial_sample=sample_id):
    global _RNA_REFS
    if _RNA_REFS:
        return _RNA_REFS
    print("Loading RDS references via R (this may take a few minutes) ...")
    with tempfile.TemporaryDirectory(prefix="spage_ref_") as tmp:
        r_script = Path(tmp) / "load_refs.R"
        r_script.write_text(R_LOAD_REFS)
        proc = subprocess.run(
            ["Rscript", str(r_script), tmp, patient, atlas_patient or "", str(REPO), spatial_sample],
            cwd=REPO, capture_output=True, text=True,
        )
        if proc.stdout:
            print(proc.stdout.rstrip())
        if proc.returncode != 0:
            if proc.stderr:
                print(proc.stderr.rstrip())
            proc.check_returncode()
        _RNA_REFS.update({key: read_mtx_ref(Path(tmp) / key) for key in RDS_REF_KEYS})
    print("RDS references loaded into memory; temporary files removed.")
    return _RNA_REFS


def load_reference(key):
    return load_rds_refs_from_r()[key]


def select_spatial_roi(coords, x_frac=ROI_X_FRAC, y_frac=ROI_Y_FRAC, in_tissue_only=True):
    c = coords.loc[coords["in_tissue"] == 1] if in_tissue_only else coords.copy()
    x, y = c["pxl_col_in_fullres"], c["pxl_row_in_fullres"]
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()
    x0 = xmin + (xmax - xmin) * x_frac[0]
    x1 = xmin + (xmax - xmin) * x_frac[1]
    y0 = ymin + (ymax - ymin) * y_frac[0]
    y1 = ymin + (ymax - ymin) * y_frac[1]
    mask = (x >= x0) & (x <= x1) & (y >= y0) & (y <= y1)
    roi = c.loc[mask]
    return roi["barcode"].tolist(), {"x0": x0, "x1": x1, "y0": y0, "y1": y1}


def plot_roi_preview(coords, roi_barcodes, roi_bounds):
    tissue = coords.loc[coords["in_tissue"] == 1]
    in_roi = tissue["barcode"].isin(roi_barcodes)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(tissue["pxl_col_in_fullres"], tissue["pxl_row_in_fullres"], s=0.05, c="lightgray", rasterized=True)
    ax.scatter(tissue.loc[in_roi, "pxl_col_in_fullres"], tissue.loc[in_roi, "pxl_row_in_fullres"], s=0.2, c="crimson", rasterized=True)
    ax.add_patch(plt.Rectangle((roi_bounds["x0"], roi_bounds["y0"]), roi_bounds["x1"] - roi_bounds["x0"], roi_bounds["y1"] - roi_bounds["y0"], fill=False, edgecolor="black", linewidth=1))
    ax.set_aspect("equal")
    ax.set_title("ROI slice (red) on capture area")
    ax.axis("off")
    plt.show()


def log_norm_cpm(df):
    totals = df.sum(axis=1).replace(0, np.nan)
    return np.log1p(df.div(totals, axis=0) * 1e6).fillna(0)


def log_norm_spatial(df):
    totals = df.sum(axis=1).replace(0, np.nan)
    return np.log1p(df.div(totals, axis=0) * totals.median()).fillna(0)


def spatial_plot_df(coords, spatial_index):
    return coords.loc[coords.index.intersection(spatial_index)]


def plot_spatial_gene(coords_df, values, title, ax=None, s=2, vmax_pct=98, vmin=None, vmax=None, show_colorbar=True):
    vals = np.asarray(values, dtype=float)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    pos = vals[vals > 0]
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    if vmin is None:
        vmin = 0.0
    if vmax is None:
        vmax = float(np.percentile(pos, vmax_pct)) if len(pos) else float(np.percentile(vals, vmax_pct)) if len(vals) else 1.0
    if vmax <= vmin:
        vmax = float(pos.max()) if len(pos) else float(vals.max()) if len(vals) else vmin + 1e-6
    if vmax <= vmin:
        vmax = vmin + 1e-6
    sc = ax.scatter(
        coords_df["pxl_col_in_fullres"], coords_df["pxl_row_in_fullres"],
        s=s, c=np.clip(vals, vmin, vmax), cmap="viridis", vmin=vmin, vmax=vmax, rasterized=True,
    )
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_title(title)
    ax.axis("off")
    if show_colorbar:
        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    return ax


def plot_missing_panel(coords_df, title, ax, s=1.2):
    ax.scatter(coords_df["pxl_col_in_fullres"], coords_df["pxl_row_in_fullres"], s=s, c="#e0e0e0", rasterized=True)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_title(f"{title}\n(not available)" if title else "(not available)")
    ax.axis("off")


def alignment_genes(spatial_df, rna_df, min_std=1e-12):
    shared = spatial_df.columns.intersection(rna_df.columns)
    spatial_std = spatial_df[shared].std(axis=0)
    rna_std = rna_df[shared].std(axis=0)
    return spatial_std[(spatial_std > min_std) & (rna_std > min_std)].index.tolist()


def prepare_spage_inputs(spatial_df, rna_df, min_cells=10):
    spatial_df = dedupe_columns(spatial_df)
    rna_df = dedupe_columns(rna_df)
    spatial_df = spatial_df.loc[spatial_df.sum(axis=1) > 0]
    rna_df = rna_df.loc[rna_df.sum(axis=1) > 0]
    rna_df = rna_df.loc[:, (rna_df > 0).sum(axis=0) >= min_cells]
    spatial_df = log_norm_spatial(spatial_df).replace([np.inf, -np.inf], 0).fillna(0)
    rna_df = log_norm_cpm(rna_df).replace([np.inf, -np.inf], 0).fillna(0)
    variable_shared = alignment_genes(spatial_df, rna_df)
    return spatial_df[variable_shared], rna_df


def spage_impute(Spatial_data, RNA_data, n_pv, genes_to_predict=None):
    if genes_to_predict is None:
        genes_to_predict = np.setdiff1d(RNA_data.columns, Spatial_data.columns)
    RNA_data_scaled = pd.DataFrame(st.zscore(RNA_data, axis=0, nan_policy="omit"), index=RNA_data.index, columns=RNA_data.columns).fillna(0)
    Spatial_data_scaled = pd.DataFrame(st.zscore(Spatial_data, axis=0, nan_policy="omit"), index=Spatial_data.index, columns=Spatial_data.columns).fillna(0)
    common = RNA_data_scaled.columns.intersection(Spatial_data_scaled.columns)
    Common_data = RNA_data_scaled[common]
    Y_train = RNA_data[genes_to_predict].fillna(0)
    Imp_Genes = pd.DataFrame(np.zeros((Spatial_data.shape[0], len(genes_to_predict))), index=Spatial_data.index, columns=genes_to_predict)
    pv_Spatial_RNA = PVComputation(n_factors=n_pv, n_pv=n_pv, dim_reduction="pca", dim_reduction_target="pca")
    pv_Spatial_RNA.fit(Common_data, Spatial_data_scaled[common])
    S = pv_Spatial_RNA.source_components_.T
    effective_n_pv = max(int(np.sum(np.diag(pv_Spatial_RNA.cosine_similarity_matrix_) > 0.3)), 1)
    S = S[:, :effective_n_pv]
    Common_data_projected = Common_data.dot(S)
    Spatial_data_projected = Spatial_data_scaled[common].dot(S)
    n_neighbors = min(50, Common_data_projected.shape[0])
    nbrs = NearestNeighbors(n_neighbors=n_neighbors, algorithm="auto", metric="cosine").fit(Common_data_projected.to_numpy())
    distances, indices = nbrs.kneighbors(Spatial_data_projected.to_numpy())
    for j in range(Spatial_data.shape[0]):
        idx = indices[j]
        d = np.asarray(distances[j], dtype=float)
        close = d < 1
        if not np.any(close):
            order = np.argsort(d)[: min(5, len(d))]
            idx, d = idx[order], d[order]
        else:
            idx, d = idx[close], d[close]
        if len(d) == 0:
            continue
        w = 1.0 / (d + 1e-8)
        Imp_Genes.iloc[j, :] = Y_train.iloc[idx].to_numpy().T @ (w / w.sum())
    return Imp_Genes.fillna(0)


def run_spage(spatial_df, rna_df, n_pv, genes_to_predict):
    genes_to_predict = [g for g in genes_to_predict if g in rna_df.columns]
    align = alignment_genes(spatial_df, rna_df)
    if len(align) > MAX_ALIGN_GENES:
        align = spatial_df[align].var(axis=0).nlargest(MAX_ALIGN_GENES).index.tolist()
    rna_cols = list(dict.fromkeys(list(align) + genes_to_predict))
    return spage_impute(spatial_df[align], rna_df[rna_cols], n_pv=min(n_pv, len(align)), genes_to_predict=genes_to_predict)


def save_fig(fig, stem, **kwargs):
    for ext in ("pdf", "jpeg"):
        path = FIG_OUT / f"{stem}.{ext}"
        fig.savefig(path, bbox_inches="tight", format=ext, dpi=300 if ext == "jpeg" else None, **kwargs)
        print(f"Wrote {path}")


def gene_values_on_plot(expr_df, gene):
    if gene not in expr_df.columns:
        return pd.Series(0.0, index=plot_df.index)
    return expr_df[gene].reindex(plot_df.index).fillna(0)


def spearman_xy(x, y, min_n=10):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < min_n:
        return np.nan
    return st.spearmanr(x[mask], y[mask]).correlation

In [ ]:
# --- 1. Load spatial ROI ---
cfg = SAMPLES[sample_id]
coords = pd.read_parquet(cfg["tissue_positions"])
roi_barcodes, roi_bounds = select_spatial_roi(coords, ROI_X_FRAC, ROI_Y_FRAC)
plot_roi_preview(coords, roi_barcodes, roi_bounds)

spatial_raw = read_10x_h5(cfg["spatial_h5"], barcodes=roi_barcodes)
coords = coords.set_index("barcode")
active_barcodes = spatial_raw.index[spatial_raw.sum(axis=1) > 0]
spatial_norm = log_norm_spatial(spatial_raw.loc[active_barcodes]).fillna(0)

ref_section = load_reference("simp_section")
spatial_data, rna_section = prepare_spage_inputs(spatial_raw.copy(), ref_section)
plot_df = spatial_plot_df(coords, spatial_norm.index.intersection(spatial_data.index))

sn_only = [g for g in rna_section.columns if g not in spatial_data.columns and not str(g).startswith(("DEPRECATED_", "MT-"))]
extra_genes = rna_section[sn_only].var(axis=0).nlargest(30).index.tolist()
IMPUTE_GENES = sorted(set(PANEL_GENES + extra_genes))
print(f"ROI bins: {spatial_data.shape[0]:,} | imputing {len(IMPUTE_GENES)} genes")

In [ ]:
# --- 2. SpaGE imputation per reference cohort ---
imputed = {}
for key in COHORT_KEYS:
    label = COHORT[key]
    print(f"Imputing with {label} ...")
    rna_raw = load_reference(key)
    spatial_in, rna = prepare_spage_inputs(spatial_raw.copy(), rna_raw)
    genes = [g for g in IMPUTE_GENES if g in rna.columns]
    missing_panel = [g for g in PANEL_GENES if g not in rna.columns]
    if missing_panel:
        print(f"  panel genes absent from reference (post-QC): {missing_panel}")
    imputed[label] = run_spage(spatial_in, rna, n_pv=N_PV, genes_to_predict=genes)
    missing_out = [g for g in PANEL_GENES if g not in imputed[label].columns]
    if missing_out:
        print(f"  panel genes missing from imputed output: {missing_out}")
print("Done.")

In [ ]:
# --- 3. Spatial maps ---
MAP_GENES = list(PANEL_GENES)
col_titles = ["Measured"] + COHORT_LABELS
n_cols, n_genes = len(col_titles), len(MAP_GENES)

fig, axes = plt.subplots(n_genes, n_cols, figsize=(2.8 * n_cols, 2.5 * n_genes), squeeze=False)
fig.subplots_adjust(top=0.90, wspace=0.65, hspace=0.35)

for i, gene in enumerate(MAP_GENES):
    ax = axes[i, 0]
    if gene in spatial_norm.columns:
        plot_spatial_gene(plot_df, gene_values_on_plot(spatial_norm, gene), "", ax=ax, s=1.5, show_colorbar=True)
    else:
        plot_missing_panel(plot_df, "", ax, s=1.5)
    ax.set_title(gene, fontsize=7)

    for j, label in enumerate(COHORT_LABELS, start=1):
        ax = axes[i, j]
        imp_df = imputed[label]
        if gene in imp_df.columns:
            plot_spatial_gene(plot_df, gene_values_on_plot(imp_df, gene), "", ax=ax, s=1.5, show_colorbar=True)
        else:
            plot_missing_panel(plot_df, "", ax, s=1.5)
        if i == 0:
            ax.set_title(label, fontsize=6)

pos = axes[0, 0].get_position()
fig.text((pos.x0 + pos.x1) / 2, pos.y1 + 0.04, "Measured", ha="center", va="bottom", fontsize=7)
fig.suptitle(f"Spatial expression — measured vs imputed ({sample_id} ROI)", y=0.99, fontsize=9)
save_fig(fig, "spatial_maps_measured_vs_imputed")
plt.show()

In [ ]:
# --- 4. Spearman concordance (imputed vs measured) ---
MIN_DETECT_BINS = 50
MEASURED_GENES = sorted(
    g for g in spatial_norm.columns
    if all(g in imp.columns for imp in imputed.values())
    and (spatial_norm[g] > 0).sum() >= MIN_DETECT_BINS
)
print(f"Genes with measured spatial signal: {len(MEASURED_GENES)}")

spearman_rows = []
for label in COHORT_LABELS:
    imp_df = imputed[label]
    idx = plot_df.index.intersection(imp_df.index)
    for gene in MEASURED_GENES:
        spearman_rows.append({
            "reference": label,
            "gene": gene,
            "spearman": spearman_xy(spatial_norm.loc[idx, gene].values, imp_df.loc[idx, gene].values),
        })
spearman_df = pd.DataFrame(spearman_rows)
spearman_df.to_csv(FIG_OUT / "imputed_vs_measured_spearman.csv", index=False)

pivot = spearman_df.pivot(index="gene", columns="reference", values="spearman").reindex(columns=COHORT_LABELS)
fig, ax = plt.subplots(figsize=(6, max(4, 0.25 * len(pivot))))
sns.heatmap(pivot, cmap="RdYlBu_r", center=0, vmin=-0.2, vmax=0.6, annot=len(pivot) <= 30, fmt=".2f", ax=ax, cbar_kws={"label": "Spearman rho"})
ax.set_title(f"Imputed vs measured spatial concordance\n{sample_id} ROI")
plt.tight_layout()
save_fig(fig, "imputed_vs_measured_spearman_heatmap")
plt.show()

summary = spearman_df.groupby("reference", observed=True)["spearman"].agg(["mean", "median", "count"]).reindex(COHORT_LABELS).reset_index()
summary.to_csv(FIG_OUT / "imputed_vs_measured_spearman_summary.csv", index=False)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(summary["reference"], summary["mean"], color=[COHORT_COLORS.get(r, "#888") for r in summary["reference"]])
ax.set_ylabel("Mean Spearman rho")
ax.set_title(f"Mean imputed vs measured concordance\n{sample_id} ROI")
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout()
save_fig(fig, "imputed_vs_measured_spearman_summary")
plt.show()

fig, ax = plt.subplots(figsize=(6, max(4, 0.2 * len(MEASURED_GENES))))
for i, gene in enumerate(MEASURED_GENES):
    for _, row in spearman_df[spearman_df["gene"] == gene].iterrows():
        ax.scatter(row["spearman"], i, color=COHORT_COLORS.get(row["reference"], "#888"), s=25, zorder=2)
ax.set_yticks(range(len(MEASURED_GENES)))
ax.set_yticklabels(MEASURED_GENES, fontsize=6)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("Spearman rho (imputed vs measured)")
ax.set_title(f"Per-gene concordance by reference\n{sample_id} ROI")
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=COHORT_COLORS.get(ref, "#888"), markersize=5, label=ref) for ref in COHORT_LABELS]
ax.legend(handles=handles, title="Reference", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
save_fig(fig, "imputed_vs_measured_spearman_genes")
plt.show()